# N27 — Race Situation Agent

This notebook builds the **Race Situation Agent**, the third sub-agent in the F1
multi-agent system. It answers two questions per lap:

- **Can we overtake?** — probability that driver X passes the car directly ahead
  within the next few laps, given the current gap, pace delta, and tyre context.
- **Is a Safety Car coming?** — probability of a SC deployment within the next
  3 laps, given the current race chaos level.

Both questions are answered by LightGBM classifiers trained in N12 and N14,
wrapped in LangChain `@tool` functions and orchestrated by a LangGraph ReAct agent.

Models loaded from `data/models/`:

| File | Source | Contents |
|---|---|---|
| `overtake_probability/lgbm_overtake_v1.pkl` | N12 | LightGBM overtake classifier |
| `overtake_probability/calibrator.pkl` | N12 | Platt calibrator (val 2024) |
| `overtake_probability/model_config.json` | N12 | Features, threshold, CAT_FEATURES |
| `safety_car_probability/lgbm_sc_v1.pkl` | N14 | LightGBM SC classifier |
| `safety_car_probability/calibrator_sc_v1.pkl` | N14 | Platt calibrator |
| `safety_car_probability/feature_list_v1.json` | N14 | Feature list (ordered) |

Output consumed by N31 Orchestrator to condition the pit strategy decision in N28.


---

## Step 0 — Setup & model loading

Imports, repo root, FastF1 cache. `RaceSituationConfig` loads both model pairs
(overtake + SC) and the circuit cluster map. All models serialised with `joblib`
(required on Windows paths containing non-ASCII characters).


In [1]:
import json, sys
from dataclasses import dataclass, field
from pathlib import Path

import fastf1
import joblib
import numpy as np
import pandas as pd

repo_root = Path.cwd()
while not (repo_root / ".git").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent


In [2]:
@dataclass
class RaceSituationConfig:
    """Runtime configuration for the Race Situation Agent.

    Loads both LightGBM model pairs (overtake from N12, SC from N14) plus their
    Platt calibrators and feature lists. Models are loaded with joblib because the
    repo path contains non-ASCII characters that break LightGBM's native save_model
    on Windows.

    overtake_threshold and sc_threshold are the classification thresholds derived
    from the Optuna/F2-score tuning in N12 and N14 respectively. threat_level
    boundaries (high_overtake, high_sc, medium_overtake, medium_sc) are agent-level
    decision rules that map raw probabilities to LOW/MEDIUM/HIGH for N31.
    """

    model_name: str = "local-model"

    # Threat-level boundaries
    high_overtake: float   = 0.80   # N12 optimal threshold — above this → HIGH
    medium_overtake: float = 0.40
    high_sc: float         = 0.30   # above this → HIGH regardless of overtake
    medium_sc: float       = 0.15

    def __post_init__(self):
        root = Path.cwd()
        while not (root / ".git").exists():
            root = root.parent

        fastf1.Cache.enable_cache(str(root / "data" / "cache" / "fastf1"))

        self.export_dir = root / "data" / "models" / "agents"
        self.export_dir.mkdir(parents=True, exist_ok=True)

        # --- Overtake model (N12) ---
        _ov_dir = root / "data" / "models" / "overtake_probability"
        self.overtake_model      = joblib.load(_ov_dir / "lgbm_overtake_v1.pkl")
        self.overtake_calibrator = joblib.load(_ov_dir / "calibrator.pkl")
        with open(_ov_dir / "model_config.json") as f:
            ov_cfg = json.load(f)
        self.overtake_features: list[str]     = ov_cfg["features"]
        self.overtake_cat_features: list[str] = ov_cfg["categorical_features"]
        self.overtake_threshold: float        = ov_cfg["optimal_threshold"]

        # --- SC model (N14) ---
        _sc_dir = root / "data" / "models" / "safety_car_probability"
        self.sc_model      = joblib.load(_sc_dir / "lgbm_sc_v1.pkl")
        self.sc_calibrator = joblib.load(_sc_dir / "calibrator_sc_v1.pkl")
        with open(_sc_dir / "feature_list_v1.json") as f:
            _sc_cfg = json.load(f)
        self.sc_features: list[str] = _sc_cfg["features"]
        self.sc_threshold: float    = _sc_cfg["best_threshold"]

        # --- Circuit cluster map (N03) — keyed by GP_Name ---
        _cluster_df = pd.read_parquet(
            root / "data" / "processed" / "circuit_clustering" / "circuit_clusters_k4.parquet"
        )
        self.circuit_cluster_map: dict = dict(
            zip(_cluster_df["GP_Name"], _cluster_df["Cluster"].astype(int))
        )

        # --- Circuit SC base rates (N13) — keyed by event_name ---
        _sc_rates_path = root / "data" / "processed" / "sc_labeled" / "sc_labeled_2023_2025.parquet"
        _sc_df = pd.read_parquet(_sc_rates_path, columns=["event_name", "circuit_sc_rate"])
        self.circuit_sc_rate_map: dict = (
            _sc_df.drop_duplicates("event_name")
            .set_index("event_name")["circuit_sc_rate"]
            .to_dict()
        )

In [3]:
CFG = RaceSituationConfig()

print(f"Overtake model     : {type(CFG.overtake_model).__name__}")
print(f"Overtake features  ({len(CFG.overtake_features)}): {CFG.overtake_features}")
print(f"Overtake cat       : {CFG.overtake_cat_features}")
print(f"Overtake threshold : {CFG.overtake_threshold:.4f}")
print()
print(f"SC model           : {type(CFG.sc_model).__name__}")
print(f"SC features        ({len(CFG.sc_features)}): {CFG.sc_features[:6]} ...")
print(f"SC threshold       : {CFG.sc_threshold:.4f}")
print()
print(f"Circuits in cluster map  : {len(CFG.circuit_cluster_map)}")
print(f"Circuits in SC rate map  : {len(CFG.circuit_sc_rate_map)}")

Overtake model     : LGBMClassifier
Overtake features  (15): ['gap_ahead_s', 'pace_delta_s', 'tyre_life_x', 'tyre_life_y', 'tyre_life_diff', 'speed_trap_delta', 'LapNumber', 'drs_window', 'compound_x', 'compound_y', 'circuit_cluster', 'gap_pace_product', 'drs_ready_gap', 'gap_trend', 'pace_delta_rolling3']
Overtake cat       : ['compound_x', 'compound_y', 'circuit_cluster']
Overtake threshold : 0.7976

SC model           : LGBMClassifier
SC features        (32): ['lap_time_mean_z', 'lap_time_std_z', 'lap_time_min_z', 'lap_time_cv', 'lap_time_trend_5', 'n_drivers'] ...
SC threshold       : 0.2335

Circuits in cluster map  : 25
Circuits in SC rate map  : 23


### Step 0 results

| Model | Type | Features | Threshold |
|---|---|---|---|
| Overtake (N12) | LGBMClassifier | 15 | 0.7976 |
| SC (N14) | LGBMClassifier | 32 | 0.2335 |

- Circuit cluster map: 25 circuits
- SC rate map: 23 circuits


---